## Imports

In [ ]:
import sys
import os
sys.path.append("../src/")

In [ ]:
import importlib
import cdsaxs
import numpy as np
from datetime import datetime

# if you are not actively developing the code, you can comment the rest of these out
importlib.reload(cdsaxs)
importlib.reload(cdsaxs.data1d)
importlib.reload(cdsaxs.data2d)
importlib.reload(cdsaxs.dataset)
importlib.reload(cdsaxs.loaders)
importlib.reload(cdsaxs.metadata)
importlib.reload(cdsaxs.plotting)
importlib.reload(cdsaxs._plotting_tools)
importlib.reload(cdsaxs.reduction)

## Metadata Location and sample name

File follows format - sample_name + "_metadata.csv"
Generate metadata file on the fly based on beamline formatting

In [ ]:
# set the path to the metadata CSV file on your computer for an SMI dataset
metadata_path = os.path.abspath("C:\\Users\\maw1\\Documents\\_RESEARCH\\_NRCCHIPS\\250309_CMSSMIBeamtime\\pass-315554\\projects\\cdsaxs\\user_data\\1M_workingCopy\\Wade\\")
data_name = "W1_15"


# construct the full path to the CSV file
file_name = data_name + "_metadata.csv"
csv_path = os.path.join(metadata_path, file_name)

# currently the General TIFF loader (with csv metadata file) is the only one implemented in the refactored code
dataset = cdsaxs.loaders.GeneralTIFFLoader(csv_path, name=file_name)


## Default values to be entered by user after initial measurements

### Sample detector distance
    Determined from NICOS and AgBh standard
    Set "sdd_cm" accordingly

### Beam center
    Determined from first grating measurement
    1. Set "find_beam_center" to true
    2. Copy x and y values into "beam_center_x_px" and "beam_center_y_px"
    3. Set "find_beam_center" to false once values are copied

### Batch run
    Once all values are determined (beam center, integration box, etc), set "batch_run" to true to skip to processed data

In [ ]:
#Default values for beamline metadata (set by user at beamline)
sdd_cm = 504.867  # SDD distance in cm
beam_center_x = 492.67
beam_center_y = 738.17

#Set find_beam_center to true to find the beam center from the data
find_beam_center = True
batch_run = False


### Update Metadata for all files loaded

In [ ]:
dataset.update_all_metadata({'sdd_cm': 504.867})

if batch_run or not find_beam_center:
    beam_center = [np.float64(beam_center_y), np.float64(beam_center_x)]
    dataset.update_all_metadata({'center_px': beam_center})

## Display all data loaded

In [ ]:
#Display all files in the dataset for easy access and copying select files
if not batch_run:
    dataset.datas

## Select Data file and show 2D data

In [ ]:
# Set data to actively work with before batch analysis

if not batch_run or find_beam_center:
    first_file = next(iter(dataset.datas)) # set by user - select the file that you want to work with
    data = dataset.datas[first_file]
    data.plot_data(show_pixels=False, log_scale=True)

## Find beam center from peaks

In [ ]:
#Locate beamcenter with peaks
if find_beam_center:
    beam_center = data.find_beam_center_from_peaks([740, 490], size_qdy_px=20, size_qdx_px=600, peak_axis='qdx', peak_params={'distance': 20, 'height': 10, 'prominence': 50}, peak_find_scale='linear')
    print(beam_center)

### Update data with new beam center

In [ ]:
#Update beamcenter at the dataset level 
if find_beam_center:
    dataset.update_all_metadata({'center_px': beam_center})

## Define integration box

In [ ]:
# define box and check peaks found within box
if not batch_run:
    qslice = data.integrate_box_of_size(7, 400, mode='sum', axis='qdy', show_plot=True, log_scale=True)

## Apply integration box to dataset and plot bowtie

In [ ]:
#Apply integraition box to the whole dataset and generate an integrated dataset (collection of integrated q slices)
cdsaxs.reduction.integrate_dataset_box_of_size(dataset, 7, 400, mode='sum', axis='qdy', in_place=True)

#Create reduced QszQsx dataset from the integrated dataset and plot it
cdsaxs.reduction.create_reduced_QszQsx(dataset, integrated_index=0)
dataset.plot_reduced_dataset()


## Select q slices and perform cuts
    Users should select and adjust these as necessary!

In [ ]:
#Generate list of q values for slicing the reduced dataset
q_values = np.arange(2, 11)*2*np.pi/1000 * 0.72

#Carry out slices
print('Slices taken at', q_values)

cdsaxs.reduction.slice_reduced_dataset(dataset, q_values=list(q_values), q_widths=0.01, in_place=True, show_plot=True)

In [ ]:
#Plot slices from the reduced dataset
dataset.plot_reduced_slices()

## Save reduced data
    Saved in the same location as the metadata
    Date and time included to prevent overwriting

In [ ]:
datetime_str = datetime.now().strftime("D%y%m%d_T%H%M%S")
save_name = os.path.join(metadata_path, data_name + '_' + datetime_str + '_reduced_results.csv')
dataset.save_reduced_slices(save_name)